In [0]:
%python
from pyspark.sql.functions import col , expr , broadcast ,current_date , current_timestamp ,max
from pyspark.sql.types import DecimalType

class EodMarketValue:

    def __init__(self):
        self.spark = spark


    def read_eod_quantity(self):

        eod_qty = self.spark.read.table("dev.spark_db.eod_quantity")
    
        return eod_qty


    def process_market_val(self,eod_qty_df):

        eod_stg_df = self.spark.read.table("dev.spark_db.eod_stg").where("as_of_dt = current_date()")

        current_quantity = (

                        eod_qty_df.alias("eq").join(broadcast(eod_stg_df).alias("es"),
                                                    on = expr("""
                                                          eq.account_number = es.account_number
                                                          and 
                                                          eq.asset_external_id = es.asset_external_id
                                                    """),
                                                    how = "inner"
                                                    )
                                              .groupBy("eq.account_number" , "eq.asset_external_id")
                                              .agg(
                                                  max(col("eq.as_of_dt")).alias("max_as_of_dt"),
                                                  max(col("eq.connection_id")).alias("max_connect_id")
                                                   )
        )

        current_quantity_detail = eod_qty_df.alias("eqd").join(
                                                          current_quantity.alias("cq"),
                                                          on = expr("""
                                                                eqd.account_number = cq.account_number
                                                                and 
                                                                eqd.asset_external_id = cq.asset_external_id
                                                                and 
                                                                as_of_dt = cq.max_as_of_dt

                                                          """),
                                                          how = 'inner'
                                                          ).select("eqd.account_number",
                                                                   "eqd.asset_external_id",
                                                                   "eqd.as_of_dt",
                                                                   "eqd.total_quantity",
                                                                   "connection_id"

                                                          )

        return current_quantity_detail


    def get_asset_rate(self , asset_list):

        asset_rate = self.spark.read.table("dev.spark_db.asset_rate")

        process_dt_rate = (asset_rate.alias("ar").join(

                                                    asset_list.alias("al"),
                                                    on = expr("""

                                                            ar.asset_external_id = al.asset_external_id
                                                            and
                                                            ar.as_of_dt = al.as_of_dt

                                                    """),
                                                    how = "inner"
                                                    ).
                                                    select("ar.asset_external_id",
                                                        "ar.asset_rate",
                                                        "al.as_of_dt")
                         )
        return process_dt_rate


    def process_eod_market_value(self,eligible_record):

        asset_list = ( eligible_record.select("asset_external_id" , current_date().alias("as_of_dt"))
                                .distinct()
        )

        asset_df = self.get_asset_rate(asset_list)

        mkt_val = ( eligible_record.alias("er").join(
                                                asset_df.alias("ad"),
                                                on = expr ("""er.asset_external_id = ad.asset_external_id""" ),
                                                how = "inner"
                                                ).
                                                where(col("total_quantity") != 0 )
                                                .select("er.account_number",
                                                        "er.asset_external_id",
                                                        current_date().alias("as_of_dt"),
                                                        (col("asset_rate") * col("total_quantity")).cast(DecimalType(18,2)).alias("mkt_value"),
                                                        "er.total_quantity",
                                                        "ad.asset_rate",
                                                        "er.connection_id",
                                                        expr("1 as eod_market_value_ver"),
                                                        current_timestamp().alias("eod_market_value_dml")
                                                )
        )

        return mkt_val

    def write_to_eod_mkt_val(self , eod_mkt_val_df):

        eod_mkt_val_df.printSchema()

        eod_mkt_val_df.write.format("delta")\
                            .mode("append")\
                            .saveAsTable("dev.spark_db.eod_market_value")

    def main(self):

        read_eod_qty = self.read_eod_quantity()

        process_market_val = self.process_market_val(read_eod_qty)

        process_eod_mkt_val = self.process_eod_market_value(process_market_val)

        self.write_to_eod_mkt_val(process_eod_mkt_val)

if __name__ == '__main__':
    emv = EodMarketValue()

    emv.main()
        


